In [27]:
import pandas as pd
import numpy as np
import os

In [28]:
from tensorflow.keras.preprocessing.image import load_img, img_to_array, ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import Sequence
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split

LOAD DATASET

In [60]:
DATASET_PATH = "dataset"
CSV_PATH = "labels_clean.csv"

df = pd.read_csv(CSV_PATH)

DEFINE CLASSES

In [61]:
classes = [
    "acne_inflammatory",
    "acne_noninflammatory",
    "hyperpigmentation",
    "wrinkles"
]

In [58]:
import pandas as pd

df = pd.read_csv("labels.csv")

# Keep only correct columns
df = df[[
    "filename",
    "acne_inflammatory",
    "acne_noninflammatory",
    "hyperpigmentation",
    "wrinkles"
]]

df.to_csv("labels_clean.csv", index=False)

print("CSV cleaned ✅")

CSV cleaned ✅


DATA AUGUMENTATION

In [62]:
datagen = ImageDataGenerator(
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

CUSTOM DATA GENERATOR

In [63]:
from tensorflow.keras.utils import Sequence

class DataGenerator(Sequence):

    def __init__(self, df, batch_size=32):
        self.df = df.reset_index(drop=True)
        self.batch_size = batch_size

    def __len__(self):
        return len(self.df) // self.batch_size

    def __getitem__(self, index):
        batch_df = self.df.iloc[index*self.batch_size:(index+1)*self.batch_size]

        images = []
        labels = []

        for i in range(len(batch_df)):
            img_path = os.path.join(DATASET_PATH, batch_df.iloc[i]['filename'])

            try:
                img = load_img(img_path, target_size=(224, 224))
                img = img_to_array(img)
                img = datagen.random_transform(img)
                img = img / 255.0

                label = batch_df.iloc[i][classes].values.astype(float)

                images.append(img)
                labels.append(label)

            except:
                print("Error:", img_path)

        return np.array(images), np.array(labels)

 Train / Validation Split

In [64]:
train_df, val_df = train_test_split(df, test_size=0.2)

CREATE GENERATORS

In [65]:
train_gen = DataGenerator(train_df, batch_size=32)
val_gen = DataGenerator(val_df, batch_size=32)

Load a pretrained model

In [66]:
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

fine tunning

In [67]:
for layer in base_model.layers[:-30]:
    layer.trainable = False
for layer in base_model.layers[-30:]:
    layer.trainable = True    


In [68]:
model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(256, activation='relu'),
Dropout(0.5),
Dense(128, activation='relu'),
Dense(len(classes), activation='sigmoid')
])

In [69]:
model.compile(
    optimizer=Adam(learning_rate=0.00005),
    loss='binary_crossentropy',
    metrics=['accuracy', 'AUC', 'Precision', 'Recall']
)

In [70]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

In [71]:
model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=25,
    callbacks=[early_stop]
)

C:\Users\HP\AppData\Roaming\Python\Python311\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Error: dataset\acne_noninflamatory/aug_1_105_0_5714_jpeg_jpg.rf.1f9eb7384edb9a08629c82f4ffde0792.jpg
Error: dataset\acne_noninflamatory/blackhead-60-_jpg.rf.b9b3da7c1438aa3557b53ca400063041.jpg
Error: dataset\acne_inflamatory/levle0_215_jpg.rf.f097e437c210f49bb324658d7aa60e23.jpg
Error: dataset\acne_inflamatory/pigmentation_0_6844_jpeg.rf.870066abef2d77af64f2ed3e927a12c9.jpg
Error: dataset\acne_noninflamatory/blackhead-23-_jpg.rf.f4bcc9c614f6166d5236aa92cf8fd67b.jpg
Error: dataset\acne_inflamatory/pigmentation_0_741_jpeg.rf.031c5d38b26dfaa1fdd7108efdc71656.jpg
Error: dataset\acne_inflamatory/pigmentation_0_5183_jpeg.rf.ed6800e5b2f056550ae56ee427f4753e.jpg
Error: dataset\acne_noninflamatory/blackhead_142_jpg.rf.381156909bfe0dc08e5773a6ffbc76eb.jpg
Error: dataset\acne_noninflamatory/levle1_68_jpg.rf.5c0fc5a21a7bed16571b9861902de3e6.jpg
Error: dataset\acne_noninflamatory/OIP-3-_jpg.rf.228b6803754151a8254f15e7792ba296.jpg
Error: dataset\acne_noninflamatory/blackhead-3-Copy_jpg.rf.9a3d6d4d7

KeyboardInterrupt: 

In [44]:
model.save("skin_model.h5")

1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
Raw predictions: [5.2927317e-06 1.1349692e-06 3.8963390e-05 9.1560065e-07]


In [54]:
top_k = 3
threshold = 0.5

indices = preds.argsort()[-top_k:][::-1]

for i in indices:
    if preds[i] > threshold:
        print(classes[i], f"{preds[i]*100:.2f}%")

In [57]:
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import load_img, img_to_array
import numpy as np

model = load_model("skin_model.h5")

classes = [
    "Acne_Inflammatory",
    "Acne_NonInflammatory",
    "Hyperpigmentation",
    "Wrinkles"
]

img = load_img("test_images/test1.jpg", target_size=(224, 224))
img = img_to_array(img)
img = img / 255.0   # ✅ IMPORTANT
img = np.expand_dims(img, axis=0)

preds = model.predict(img)[0]

print("Predictions:", preds)

top_k = 2
indices = preds.argsort()[-top_k:][::-1]

for i in indices:
    print(classes[i], f"{preds[i]*100:.2f}%")

1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
Predictions: [5.2927317e-06 1.1349692e-06 3.8963390e-05 9.1560065e-07]
Hyperpigmentation 0.00%
Acne_Inflammatory 0.00%


In [55]:
print("Predictions:", preds)

Predictions: [5.2927317e-06 1.1349692e-06 3.8963390e-05 9.1560065e-07]


In [56]:
model.summary()


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,619,334 (9.99 MB)

 Trainable params: 1,887,748 (7.20 MB)

 Non-trainable params: 731,584 (2.79 MB)

 Optimizer params: 2 (12.00 B)